In [1]:
%pip install -e "/content/drive/MyDrive/BUSI-XAI"

Obtaining file:///content/drive/MyDrive/BUSI-XAI
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for busi-xai (pyproject.toml) ... done
  Created wheel for busi-xai: filename=busi_xai-0.1.0-0.editable-py3-none-any.whl size=1167 sha256=be978d5aa4f55220eec3e0019398de2c222f693995948def22995e9db3ccbd99
  Stored in directory: /tmp/pip-ephem-wheel-cache-ji9rr6as/wheels/bc/96/6a/14ad5b6a03036f6f181aa62fbcb8d63ac3b6cb1da04fcdcb1e
Successfully built busi-xai


# Model preparation

## Inspect Split schema

In [ ]:
import pandas as pd

from busi_xai.config import SPLITS_DIR

train_df = pd.read_csv(SPLITS_DIR / "train.csv")
validation_df = pd.read_csv(SPLITS_DIR / "validation.csv")
test_df = pd.read_csv(SPLITS_DIR / "test.csv")

print("Train columns:")
print(train_df.columns.tolist())

print("\nTrain shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)

print("\nFirst train record:")
display(train_df.head(1))

Train columns:
['image_id', 'class_name', 'source_image', 'processed_image', 'processed_mask', 'num_masks', 'mask_paths', 'image_width', 'image_height', 'mask_width', 'mask_height', 'mask_empty', 'lesion_pixels', 'lesion_area_ratio', 'image_sha256', 'include_in_experiment', 'exclusion_reason', 'split']

Train shape: (451, 18)
Validation shape: (97, 18)
Test shape: (97, 18)

First train record:


,image_id,class_name,source_image,processed_image,processed_mask,num_masks,mask_paths,image_width,image_height,mask_width,mask_height,mask_empty,lesion_pixels,lesion_area_ratio,image_sha256,include_in_experiment,exclusion_reason,split
0,benign (43),benign,/content/drive/MyDrive/BUSI-XAI/data/raw/benig...,/content/drive/MyDrive/BUSI-XAI/data/processed...,/content/drive/MyDrive/BUSI-XAI/data/processed...,1,/content/drive/MyDrive/BUSI-XAI/data/raw/benig...,554,495,554,495,False,2331,0.0085,39f6dfa89a464e9846424e460f9e40d55a527d4a2c7e80...,True,NaN,train


## Prepare model datasets

In [ ]:
from busi_xai.data import (
    BUSIDataset,
    build_image_transform,
    compute_class_weights,
)

train_dataset = BUSIDataset(
    dataframe=train_df,
    image_column="processed_image",
    class_column="class_name",
    mask_column="processed_mask",
    image_transform=build_image_transform(train=True),
)

validation_dataset = BUSIDataset(
    dataframe=validation_df,
    image_column="processed_image",
    class_column="class_name",
    mask_column="processed_mask",
    image_transform=build_image_transform(train=False),
)

test_dataset = BUSIDataset(
    dataframe=test_df,
    image_column="processed_image",
    class_column="class_name",
    mask_column="processed_mask",
    image_transform=build_image_transform(train=False),
)

print("Datasets created successfully.")
print("Train:", len(train_dataset))
print("Validation:", len(validation_dataset))
print("Test:", len(test_dataset))

Datasets created successfully.
Train: 451
Validation: 97
Test: 97


## Check data sample


In [ ]:
sample = train_dataset[0]

print("Image shape:", sample["image"].shape)
print("Image dtype:", sample["image"].dtype)
print("Label:", sample["label"].item())
print("Mask shape:", sample["mask"].shape)
print("Mask dtype:", sample["mask"].dtype)
print("Mask unique values:", sample["mask"].unique().tolist())

Image shape: torch.Size([3, 224, 224])
Image dtype: torch.float32
Label: 0
Mask shape: torch.Size([1, 224, 224])
Mask dtype: torch.float32
Mask unique values: [0.0, 1.0]


## Calculate class weights

In [ ]:
class_weights = compute_class_weights(
    train_df,
    class_column="class_name",
)

print("Class counts:")
print(train_df["class_name"].value_counts())

print("\nClass weights:")
print(class_weights)

Class counts:
class_name
benign       305
malignant    146
Name: count, dtype: int64

Class weights:
tensor([0.7393, 1.5445])


## Create dataloaders

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

## Validate dataloaders

In [ ]:
batch = next(iter(train_loader))

print("Image batch:", batch["image"].shape)
print("Label batch:", batch["label"].shape)
print("Mask batch:", batch["mask"].shape)
print("Labels:", batch["label"][:10].tolist())

Image batch: torch.Size([32, 3, 224, 224])
Label batch: torch.Size([32])
Mask batch: torch.Size([32, 1, 224, 224])
Labels: [0, 0, 1, 1, 0, 0, 1, 0, 0, 0]


In [ ]:
import torch
# Check training augmentation over multiple calls
train_samples = [
    train_dataset[0]["image"]
    for _ in range(10)
]

unique_train_samples = sum(
    not torch.equal(train_samples[0], sample)
    for sample in train_samples[1:]
)

# Validation should remain deterministic
val_samples = [
    validation_dataset[0]["image"]
    for _ in range(10)
]

all_validation_same = all(
    torch.equal(val_samples[0], sample)
    for sample in val_samples[1:]
)

print("Different training augmentations observed:", unique_train_samples)
print("All validation samples identical:", all_validation_same)

Different training augmentations observed: 2
All validation samples identical: True


# Validate model

## Validate model builder

In [1]:
import torch

from busi_xai.model import build_resnet50


model = build_resnet50()

print(model)
print()
print("Final classifier:", model.fc)
print("Input features:", model.fc.in_features)
print("Output classes:", model.fc.out_features)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 157MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

## Validate Classification Loss configuration

In [ ]:
import torch

from busi_xai.model import build_classification_loss


class_weights = torch.tensor(
    [0.7393, 1.5445],
    dtype=torch.float32,
)

criterion = build_classification_loss(class_weights)

print("Loss function:", criterion)
print("Class weights:", criterion.weight)

Loss function: CrossEntropyLoss()
Class weights: tensor([0.7393, 1.5445])


## Validate optimizer

In [ ]:
from busi_xai.model import build_optimizer


optimizer = build_optimizer(
    model,
    learning_rate=1e-4,
    weight_decay=1e-4,
)

print("Optimizer:", optimizer)
print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Weight decay:", optimizer.param_groups[0]["weight_decay"])

Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0.0001
)
Learning rate: 0.0001
Weight decay: 0.0001


## Validate checkpoint saving

In [ ]:
from pathlib import Path
import torch

from busi_xai.config import MODELS_DIR, BEST_MODEL_FILENAME
from busi_xai.model import (
    build_resnet50,
    build_optimizer,
    save_best_checkpoint
)


model = build_resnet50()
optimizer = build_optimizer(model)

MODELS_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_path = MODELS_DIR / BEST_MODEL_FILENAME

save_best_checkpoint(
    model=model,
    optimizer=optimizer,
    epoch=1,
    validation_loss=0.75,
    path=checkpoint_path,
)

print("Checkpoint exists:", checkpoint_path.exists())
print("Checkpoint path:", checkpoint_path)

Checkpoint exists: True
Checkpoint path: /content/drive/MyDrive/BUSI-XAI/outputs/models/resnet50_best.pt


In [ ]:
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
)

assert "epoch" in checkpoint
assert "model_state_dict" in checkpoint
assert "optimizer_state_dict" in checkpoint
assert "validation_loss" in checkpoint

print("Checkpoint validation passed.")

Checkpoint validation passed.


## Test random seeding

In [ ]:
from busi_xai.config import RANDOM_SEED
from busi_xai.model import set_random_seed


set_random_seed(RANDOM_SEED)

print("Random seed:", RANDOM_SEED)
print("Reproducibility setup validation passed.")

Random seed: 42
Reproducibility setup validation passed.


# Final Model

## Imports

In [ ]:
import torch

from busi_xai.config import (
    BATCH_SIZE,
    LEARNING_RATE,
    WEIGHT_DECAY,
    NUM_EPOCHS,
    MODELS_DIR,
    BEST_MODEL_FILENAME,
    METRICS_DIR
)
from busi_xai.data import compute_class_weights
from busi_xai.model import (
    build_resnet50,
    build_classification_loss,
    build_optimizer,
    train_one_epoch,
    validate_one_epoch,
    save_best_checkpoint,
    train_model,
    load_checkpoint,
    evaluate_model,
    set_random_seed,
)
# Define device explicitly.
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

## Model train 1 epoch

In [ ]:


# Check GPU
print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))

# Create fresh model, loss, and optimizer.
model = build_resnet50().to(device)

criterion = build_classification_loss(
    class_weights=class_weights.to(device),
)

optimizer = build_optimizer(
    model=model,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

# Run exactly one training epoch.
train_loss, train_accuracy = train_one_epoch(
    model=model,
    loader=train_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
)

# Run exactly one validation epoch.
validation_loss, validation_accuracy = validate_one_epoch(
    model=model,
    loader=validation_loader,
    criterion=criterion,
    device=device,
)

torch.cuda.synchronize()
elapsed = time.perf_counter() - start_time

print(f"Training epoch time: {elapsed:.2f} seconds")
print(f"Training epoch time: {elapsed / 60:.2f} minutes")
print(f"Smoke-test train loss: {train_loss:.4f}")
print(f"Smoke-test train accuracy: {train_accuracy:.4f}")
print(f"Smoke-test validation loss: {validation_loss:.4f}")
print(f"Smoke-test validation accuracy: {validation_accuracy:.4f}")

assert train_loss > 0
assert validation_loss > 0
assert 0.0 <= train_accuracy <= 1.0
assert 0.0 <= validation_accuracy <= 1.0

print("Training smoke test passed.")

Smoke-test train loss: 0.6724
Smoke-test train accuracy: 0.6319
Smoke-test validation loss: 0.6502
Smoke-test validation accuracy: 0.7526
Training smoke test passed.


## Train Model (20 epochs)

In [ ]:
from busi_xai.config import (
    RANDOM_SEED,
    NUM_EPOCHS,
    LEARNING_RATE,
    WEIGHT_DECAY,
    MODELS_DIR,
    METRICS_DIR,
    BEST_MODEL_FILENAME,
)

checkpoint_path = MODELS_DIR / BEST_MODEL_FILENAME
history_path = METRICS_DIR / "training_history.csv"

history, best_epoch, best_validation_loss = train_model(
    train_loader=train_loader,
    validation_loader=validation_loader,
    class_weights=class_weights,
    device=device,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    checkpoint_path=checkpoint_path,
    history_path=history_path,
    seed=RANDOM_SEED,
)

print()
print("Training completed.")
print("Best epoch:", best_epoch)
print("Best validation loss:", best_validation_loss)
print("Training history:", history_path)
print("Best checkpoint:", checkpoint_path)

Epoch 01/20 | Train Loss: 0.6623 | Train Acc: 0.5920 | Val Loss: 0.6627 | Val Acc: 0.7216epoch time: 74.8795is best epoch: True
  → Best model checkpoint updated.
Epoch 02/20 | Train Loss: 0.4999 | Train Acc: 0.8359 | Val Loss: 0.5863 | Val Acc: 0.7320epoch time: 16.6466is best epoch: True
  → Best model checkpoint updated.
Epoch 03/20 | Train Loss: 0.2970 | Train Acc: 0.9135 | Val Loss: 0.4936 | Val Acc: 0.8041epoch time: 17.8120is best epoch: True
  → Best model checkpoint updated.
Epoch 04/20 | Train Loss: 0.1811 | Train Acc: 0.9446 | Val Loss: 0.4606 | Val Acc: 0.8557epoch time: 9.9296is best epoch: True
  → Best model checkpoint updated.
Epoch 05/20 | Train Loss: 0.0851 | Train Acc: 0.9823 | Val Loss: 0.4570 | Val Acc: 0.8660epoch time: 17.6175is best epoch: True
  → Best model checkpoint updated.
Epoch 06/20 | Train Loss: 0.0566 | Train Acc: 0.9889 | Val Loss: 0.3743 | Val Acc: 0.9175epoch time: 17.7562is best epoch: True
  → Best model checkpoint updated.
Epoch 07/20 | Train Los

# Check created history

In [ ]:
import pandas as pd
import torch

from busi_xai.model import build_resnet50, load_checkpoint

history_path = METRICS_DIR / "training_history.csv"
checkpoint_path = MODELS_DIR / BEST_MODEL_FILENAME

# Load training history.
history_df = pd.read_csv(history_path)

print("History rows:", len(history_df))
print("History columns:", list(history_df.columns))

# Identify best recorded epoch.
best_row = history_df.loc[
    history_df["validation_loss"].idxmin()
]

print("Best epoch from history:", int(best_row["epoch"]))
print(
    "Best validation loss from history:",
    best_row["validation_loss"],
)

# Load checkpoint.
model = build_resnet50().to(device)

checkpoint = load_checkpoint(
    model=model,
    checkpoint_path=checkpoint_path,
    device=device,
)

print("Checkpoint epoch:", checkpoint["epoch"])
print(
    "Checkpoint validation loss:",
    checkpoint["validation_loss"],
)

# Validate history.
assert len(history_df) == NUM_EPOCHS

assert int(best_row["epoch"]) == checkpoint["epoch"]

assert abs(
    best_row["validation_loss"]
    - checkpoint["validation_loss"]
) < 1e-8

assert checkpoint["epoch"] == 8

print("Training artifact validation passed.")

History rows: 20
History columns: ['epoch', 'train_loss', 'train_accuracy', 'validation_loss', 'validation_accuracy', 'epoch_time_seconds', 'best_checkpoint']
Best epoch from history: 8
Best validation loss from history: 0.314631738436988
Checkpoint epoch: 8
Checkpoint validation loss: 0.314631738436988
Training artifact validation passed.


# Model Evaluation

In [ ]:
# Load the validated best checkpoint.
model = build_resnet50().to(device)

checkpoint = load_checkpoint(
    model=model,
    checkpoint_path=checkpoint_path,
    device=device,
)

print("Loaded checkpoint epoch:", checkpoint["epoch"])
print(
    "Loaded checkpoint validation loss:",
    checkpoint["validation_loss"],
)

# Final held-out test evaluation.
test_results = evaluate_model(
    model=model,
    loader=test_loader,
    device=device,
)

print()
print("Test samples:", len(test_results["labels"]))
print("Test accuracy:", test_results["accuracy"])

Loaded checkpoint epoch: 8
Loaded checkpoint validation loss: 0.314631738436988

Test samples: 97
Test accuracy: 0.8969072103500366


# Generate complete classification metrics

In [ ]:
import numpy as np
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

labels = test_results["labels"].numpy()
predictions = test_results["predictions"].numpy()

# Confusion matrix.
cm = confusion_matrix(
    labels,
    predictions,
    labels=[0, 1],
)

tn, fp, fn, tp = cm.ravel()

# Overall metrics.
accuracy = accuracy_score(labels, predictions)

precision = precision_score(
    labels,
    predictions,
    zero_division=0,
)

recall = recall_score(
    labels,
    predictions,
    zero_division=0,
)

f1 = f1_score(
    labels,
    predictions,
    zero_division=0,
)

specificity = tn / (tn + fp)

print("Confusion matrix:")
print(cm)

print()
print(f"Accuracy:    {accuracy:.4f}")
print(f"Precision:   {precision:.4f}")
print(f"Recall:      {recall:.4f}")
print(f"F1-score:    {f1:.4f}")
print(f"Specificity: {specificity:.4f}")

print()
print("Classification report:")
print(
    classification_report(
        labels,
        predictions,
        target_names=["benign", "malignant"],
        zero_division=0,
    )
)

Confusion matrix:
[[61  5]
 [ 5 26]]

Accuracy:    0.8969
Precision:   0.8387
Recall:      0.8387
F1-score:    0.8387
Specificity: 0.9242

Classification report:
              precision    recall  f1-score   support

      benign       0.92      0.92      0.92        66
   malignant       0.84      0.84      0.84        31

    accuracy                           0.90        97
   macro avg       0.88      0.88      0.88        97
weighted avg       0.90      0.90      0.90        97



In [ ]:
import json

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

labels = test_results["labels"].numpy()
predictions = test_results["predictions"].numpy()

# Confusion matrix.
cm = confusion_matrix(
    labels,
    predictions,
    labels=[0, 1],
)

tn, fp, fn, tp = cm.ravel()

# Metrics.
accuracy = accuracy_score(labels, predictions)

precision = precision_score(
    labels,
    predictions,
    zero_division=0,
)

recall = recall_score(
    labels,
    predictions,
    zero_division=0,
)

f1 = f1_score(
    labels,
    predictions,
    zero_division=0,
)

specificity = tn / (tn + fp)

# Per-class metrics.
report = classification_report(
    labels,
    predictions,
    target_names=["benign", "malignant"],
    output_dict=True,
    zero_division=0,
)

metrics = {
    "model": "ResNet-50",
    "dataset": "BUSI",
    "test_samples": int(len(labels)),
    "checkpoint_epoch": int(checkpoint["epoch"]),
    "validation_loss_at_checkpoint": float(
        checkpoint["validation_loss"]
    ),
    "accuracy": float(accuracy),
    "precision_malignant": float(precision),
    "recall_malignant": float(recall),
    "f1_malignant": float(f1),
    "specificity_benign": float(specificity),
    "confusion_matrix": {
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
    },
    "classification_report": report,
}

metrics_path = METRICS_DIR / "test_classification_metrics.json"

with metrics_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metrics,
        file,
        indent=2,
    )

print("Final classification metrics saved to:")
print(metrics_path)

Final classification metrics saved to:
/content/drive/MyDrive/BUSI-XAI/outputs/metrics/test_classification_metrics.json
